In [2]:
from pathlib import Path
import pandas as pd
import re

# ========= paths =========
input_csv = Path(
    r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/employment_tables/ethnicity_employment_shares_prework.csv"
)
output_tex = Path(
    r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/employment_tables/ethnicity_employment_shares_prework.tex"
)
# ===============================


def latex_escape(text):
    """Escape special characters for LaTeX."""
    if pd.isna(text):
        return ""

    text = str(text)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    pattern = re.compile("|".join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group(0)], text)


# 讀入 CSV
df = pd.read_csv(input_csv)

# 保留需要的欄位
cols = [
    "ethn_group",
    "p_reduction",
    "p_jobloss",
    "p_keyworker",
    "p_selfemp",
    "n",
]

missing = [c for c in cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV 缺少欄位: {missing}")

df = df[cols].copy()

# 數值欄位轉為 numeric
num_cols = ["p_reduction", "p_jobloss", "p_keyworker", "p_selfemp", "n"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 若 CSV 未先排好，可在此指定族群順序
ethnicity_order = [
    "British/English/Scottish/Welsh/Northern Irish",
    "Indian",
    "Pakistani",
    "Bangladeshi",
    "African",
    "Caribbean",
    "Any other white background",
]

df["ethn_group"] = pd.Categorical(
    df["ethn_group"],
    categories=ethnicity_order,
    ordered=True,
)
df = df.sort_values("ethn_group").reset_index(drop=True)

# LaTeX escape
df["ethn_group"] = df["ethn_group"].astype(str).map(latex_escape)

# 產生 LaTeX table
lines = [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Employment Outcomes by Ethnicity}",
    r"\label{tab:ethnicity_employment}",
    r"\begin{threeparttable}",
    r"\footnotesize",
    r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.8cm}rrrrr}",
    r"\toprule",
    r"Ethnicity & Reduction & Job loss & Keyworker & Self-employed & N \\",
    r"\midrule",
]

for _, row in df.iterrows():
    eth = row["ethn_group"]
    red = "" if pd.isna(row["p_reduction"]) else f'{row["p_reduction"]:.2f}'
    job = "" if pd.isna(row["p_jobloss"]) else f'{row["p_jobloss"]:.2f}'
    key = "" if pd.isna(row["p_keyworker"]) else f'{row["p_keyworker"]:.2f}'
    sel = "" if pd.isna(row["p_selfemp"]) else f'{row["p_selfemp"]:.2f}'
    n = "" if pd.isna(row["n"]) else f'{int(round(row["n"])):,}'

    lines.append(f"{eth} & {red} & {job} & {key} & {sel} & {n} \\\\")

lines.extend([
    r"\bottomrule",
    r"\end{tabular*}",
    r"\begin{tablenotes}[flushleft]",
    r"\footnotesize",
    (
        r"\item Notes: This table reports weighted percentages by ethnicity among "
        r"respondents who were working before the pandemic (\(prework=1\)). "
        r"Reduction, Job loss, Keyworker, and Self-employed are all expressed as "
        r"shares of pre-pandemic workers within each ethnic group. \(N\) denotes "
        r"the unweighted sample size. Percentages are weighted using the CA Covid "
        r"survey weights."
    ),
    r"\end{tablenotes}",
    r"\end{threeparttable}",
    r"\end{table}",
])

latex_table = "\n".join(lines)

# 寫出 .tex
output_tex.write_text(latex_table, encoding="utf-8")

print(f"LaTeX table saved to: {output_tex}")
print()
print(latex_table)

LaTeX table saved to: /Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/employment_tables/ethnicity_employment_shares_prework.tex

\begin{table}[htbp]
\centering
\caption{Employment Outcomes by Ethnicity}
\label{tab:ethnicity_employment}
\begin{threeparttable}
\footnotesize
\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.8cm}rrrrr}
\toprule
Ethnicity & Reduction & Job loss & Keyworker & Self-employed & N \\
\midrule
British/English/Scottish/Welsh/Northern Irish & 46.77 & 4.06 & 29.25 & 11.55 & 7,939 \\
Indian & 30.91 & 6.87 & 37.00 & 12.55 & 135 \\
Pakistani & 50.50 & 8.52 & 24.02 & 9.37 & 81 \\
Bangladeshi & 34.63 & 18.88 & 39.36 & 8.30 & 37 \\
African & 48.10 & 9.47 & 29.11 & 1.75 & 51 \\
Caribbean & 46.14 & 2.70 & 24.46 & 4.32 & 62 \\
Any other white background & 50.80 & 3.78 & 19.07 & 21.32 & 274 \\
\bottomrule
\end{tabular*}
\begin{tablenotes}[flushleft]
\footnotesize
\item Notes: This table reports weighted percentages by ethnicity among respondents w